In [1]:
# --- CELL 1: Setup & Imports ---
import sys
import os
import numpy as np

# Add the parent directory to Python's path so we can import from src/
sys.path.append(os.path.abspath('..'))

from src.rl_env import WaterTreatmentEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [ ]:
# --- CELL 2: Test the Environment ---
# Initialize the environment
env = WaterTreatmentEnv()

# Reset to starting state
obs, info = env.reset()
total_reward = 0

print("🎲 Testing with a RANDOM Operator...")
for step in range(20):
    # Take a random action: [Pump, Valve]
    action = env.action_space.sample()
    
    # Extract current state for logging before it changes
    tank, inflow, turb, ph, chem = obs
    is_contaminated = (chem == 1.0) or (ph < 6.5) or (ph > 8.5) or (turb > 150)
    water_status = "⚠️ CONTAMINATED" if is_contaminated else "✅ CLEAN"
    
    # Apply action to environment
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    
    pump_action, valve_action = action
    pump_text = "🟢 ON " if pump_action == 1 else "🔴 OFF"
    valve_text = "RECYCLE" if valve_action == 1 else "MAIN   "
    
    print(f"Step {step+1:02d} | Water: {water_status.ljust(15)} | Valve: {valve_text} | Pump: {pump_text} | Tank: {obs[0]:.1f}% | Reward: {reward}")
    
    if terminated or truncated:
        break

print(f"\nFinal Score (Random): {total_reward}")

In [2]:
# --- CELL 3: Train the PPO Agent ---
import sys
import os
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# Add path so it can find your environment
sys.path.append(os.path.abspath('..'))
from src.rl_env import WaterTreatmentEnv

print("🧠 Training the Neural WaterNet Agent... (This might take a minute)")

# Wrap the environment for Stable Baselines
vec_env = DummyVecEnv([lambda: WaterTreatmentEnv()])

# We can return to normal learning speeds and standard curiosity
model = PPO(
    "MlpPolicy", 
    vec_env, 
    verbose=1, 
    learning_rate=0.001,  # Standard learning speed
    ent_coef=0.05         # Normal curiosity (just enough to keep it trying new things)
)

# 100,000 steps is more than enough now that the game isn't rigged!
model.learn(total_timesteps=100000)

print("\n✅ Training Complete!")

🧠 Training the Neural WaterNet Agent... (This might take a minute)
Using cpu device


c:\Users\Ilyas\Documents\VIC_Internship\NeuralWaterNet\NeuralWaterNetvenv\lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\Ilyas\Documents\VIC_Internship\NeuralWaterNet\NeuralWaterNetvenv\lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


-----------------------------
| time/              |      |
|    fps             | 999  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 724        |
|    iterations           | 2          |
|    time_elapsed         | 5          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.01595847 |
|    clip_fraction        | 0.238      |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.37      |
|    explained_variance   | 0.000385   |
|    learning_rate        | 0.001      |
|    loss                 | 2.95e+05   |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.027     |
|    value_loss           | 6.31e+05   |
----------------------------------------
------------------------------------------
| time/  

In [3]:
# --- CELL 4: Evaluate the Trained Agent ---
print("🏆 Testing the TRAINED AI Operator with Valve Control...")

from stable_baselines3.common.evaluation import evaluate_policy
from src.rl_env import WaterTreatmentEnv 

# Evaluate
mean_reward, std_reward = evaluate_policy(model, vec_env, n_eval_episodes=5)
print(f"Average Score over 5 episodes: {mean_reward:.2f} +/- {std_reward:.2f}\n")

# Run a sample episode
env = WaterTreatmentEnv() # <-- This line fixes the NameError
obs, info = env.reset()
for step in range(20):
    action, _states = model.predict(obs, deterministic=True)
    
    # action is now an array: [pump_action, valve_action]
    pump_action, valve_action = action
    
    # Extract current state for logging before it changes
    tank, inflow, turb, ph, chem = obs
    is_contaminated = (chem == 1.0) or (ph < 6.5) or (ph > 8.5) or (turb > 150)
    water_status = "⚠️ CONTAMINATED" if is_contaminated else "✅ CLEAN"
    
    # Apply action
    obs, reward, terminated, truncated, info = env.step(action)
    
    pump_text = "ON " if pump_action == 1 else "OFF"
    valve_text = "RECYCLE" if valve_action == 1 else "MAIN   "
    
    print(f"Step {step+1:02d} | Water: {water_status.ljust(15)} | Valve: {valve_text} | Pump: {pump_text} | Tank: {obs[0]:.1f}% | Reward: {reward}")
    
    if terminated or truncated:
        break

🏆 Testing the TRAINED AI Operator with Valve Control...


c:\Users\Ilyas\Documents\VIC_Internship\NeuralWaterNet\NeuralWaterNetvenv\lib\site-packages\stable_baselines3\common\evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Average Score over 5 episodes: 2179.80 +/- 247.98

Step 01 | Water: ✅ CLEAN         | Valve: MAIN    | Pump: OFF | Tank: 51.5% | Reward: 30.0
Step 02 | Water: ✅ CLEAN         | Valve: MAIN    | Pump: OFF | Tank: 57.2% | Reward: 30.0
Step 03 | Water: ⚠️ CONTAMINATED | Valve: RECYCLE | Pump: ON  | Tank: 47.2% | Reward: 29.0
Step 04 | Water: ✅ CLEAN         | Valve: MAIN    | Pump: ON  | Tank: 40.8% | Reward: 29.0
Step 05 | Water: ✅ CLEAN         | Valve: MAIN    | Pump: OFF | Tank: 43.2% | Reward: 30.0
Step 06 | Water: ⚠️ CONTAMINATED | Valve: MAIN    | Pump: OFF | Tank: 45.6% | Reward: -90.0
Step 07 | Water: ✅ CLEAN         | Valve: MAIN    | Pump: OFF | Tank: 49.3% | Reward: 30.0
Step 08 | Water: ✅ CLEAN         | Valve: MAIN    | Pump: ON  | Tank: 44.3% | Reward: 29.0
Step 09 | Water: ⚠️ CONTAMINATED | Valve: RECYCLE | Pump: ON  | Tank: 34.3% | Reward: 29.0
Step 10 | Water: ✅ CLEAN         | Valve: MAIN    | Pump: OFF | Tank: 36.4% | Reward: 30.0
Step 11 | Water: ✅ CLEAN         | Val

c:\Users\Ilyas\Documents\VIC_Internship\NeuralWaterNet\NeuralWaterNetvenv\lib\site-packages\gymnasium\spaces\box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
c:\Users\Ilyas\Documents\VIC_Internship\NeuralWaterNet\NeuralWaterNetvenv\lib\site-packages\gymnasium\spaces\box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [4]:
# --- CELL 5: Save the Model ---
import os
os.makedirs('models/rl', exist_ok=True)
model_path = "models/rl/ppo_waternet"
model.save(model_path)
print(f"✅ RL Agent saved successfully to: {model_path}.zip")

✅ RL Agent saved successfully to: models/rl/ppo_waternet.zip
